#  ⭐ `dim_indicacao_meddra`


In [1]:
%run ../../config/bootstrap.py


from pathlib import Path
from utils import get_project_root, fuzzy_merge

import re
import pandas as pd
from unidecode import unidecode

In [2]:
project_root = get_project_root() 

# 🥉 Bronze 

Raw Data
as is production, low quality

In [3]:
path = Path(project_root) / "data/01_bronze/Medicamentos/Medicamentos.parquet"
bronze = pd.read_parquet(path)
bronze["INDICACAO_MEDDRA"] = bronze["INDICACAO_MEDDRA"].fillna("DESCONHECIDO")

bronze = pd.concat(
    [
        (
            bronze[["INDICACAO_MEDDRA"]]
            .value_counts()
            .rename("FREQUENCIA_REGISTROS")
            .reset_index()
        ),
        (
            bronze[["INDICACAO_RELATADA_NOTIFICADOR_INICIAL"]].rename(columns={"INDICACAO_RELATADA_NOTIFICADOR_INICIAL": "INDICACAO_MEDDRA"})
            .value_counts()
            .rename("FREQUENCIA_REGISTROS")
            .reset_index()
        ),
    ],
    ignore_index=True,
)

bronze.to_csv(
    Path(project_root) / "data/01_bronze/Medicamentos/Medicamentos_INDICACAO_MEDDRA.csv",
    sep=";",
    index=False,
)

In [5]:
bronze.head()

,INDICACAO_MEDDRA,FREQUENCIA_REGISTROS
0,DESCONHECIDO,325534
1,Uso de medicamento para indicação desconhecida,19595
2,Produto usado para indicação desconhecida,16515
3,Doença de Crohn,6875
4,Artrite reumatoide,6207


In [4]:
bronze.INDICACAO_MEDDRA.nunique()

67778

# 🥈 Silver

normalized data, medium quality


In [5]:
dim_soc_llt = pd.read_parquet(Path(project_root) / "data/03_gold/dim_soc/dim_soc.parquet")
dim_soc_llt.head()

,PK_SOC,SOC,HLGT,HLT,PT,PK_LLT,REACAO_EVTO_ADVERSO_MEDDRA_LLT
0,26,Circunstâncias sociais,Fatores relacionados ao gênero,Circunstâncias relacionadas à gravidez,Amamentação,1,Amamentação
1,26,Circunstâncias sociais,Fatores relacionados ao gênero,Questões de sexualidade,Abstinência sexual,2,Abstinência sexual
2,26,Circunstâncias sociais,Fatores relacionados ao gênero,Questões de sexualidade,Atividade sexual aumentada,3,Atividade sexual aumentada
3,26,Circunstâncias sociais,Fatores relacionados ao gênero,Questões de sexualidade,Não consumação,4,Não consumação
4,26,Circunstâncias sociais,Fatores relacionados à idade,Questões relacionadas à idade,Bebê,5,Neonato


In [10]:
dim_soc_llt.shape

(18549, 8)

In [7]:
silver = pd.read_csv(Path(project_root) / "data/01_bronze/Medicamentos/Medicamentos_INDICACAO_MEDDRA.csv", sep=";") ## Manual Normalization
 
silver.columns

Index(['INDICACAO_MEDDRA', 'FREQUENCIA_REGISTROS'], dtype='object')

In [11]:
silver.shape

(69945, 2)

In [ ]:
dim_soc_llt['INDICACAO_MEDDRA'] = dim_soc_llt['REACAO_EVTO_ADVERSO_MEDDRA_LLT']
hist_silver = fuzzy_merge(
    dim_soc_llt[['INDICACAO_MEDDRA','PK_LLT']],
    silver,
    on="INDICACAO_MEDDRA",
    threshold=95,
    suffix="",
    dedupe_on=False,
) 

In [ ]:
hist_silver.shape

In [ ]:
hist_silver.head()

# 🥇 Gold
